# Customer Segmentation Platform — Exploratory Data Analysis

This notebook performs EDA and data-quality checks before feature engineering and clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully")


## 1. Load Dataset

In [ ]:
df = pd.read_csv("../data/customer_data.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()


## 2. Dataset Dimensions

In [ ]:
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]:,}")


## 3. Column Names

In [ ]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


## 4. Data Types and Structure

In [ ]:
df.info()

## 5. Missing Values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing if not missing.empty else "No missing values found.")


## 6. Missing-Value Percentage

In [ ]:
missing_pct = (df.isnull().mean() * 100)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(missing_pct if not missing_pct.empty else "No missing values found.")


## 7. Duplicate Rows

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum():,}")


## 8. Duplicate Customer IDs

In [ ]:
print(f"Duplicate customer IDs: {df['customer_id'].duplicated().sum():,}")


## 9. Unique Customers

In [ ]:
print(f"Unique customers: {df['customer_id'].nunique():,}")


## 10. Confirm Customer-Level Grain

In [ ]:
if len(df) == df["customer_id"].nunique():
    print("✓ One row represents one unique customer")
else:
    print("⚠ Multiple rows exist for some customers")


## 11. Statistical Summary

In [ ]:
df.describe().T

## 12. Numerical Columns

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
print(f"Number of numerical columns: {len(numeric_cols)}")
print(list(numeric_cols))


## 13. Categorical Columns

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
print(f"Number of categorical columns: {len(categorical_cols)}")
print(list(categorical_cols))


# Data Quality Checks

## 14. Age Validation

In [ ]:
invalid_age = df[(df["age"] < 18) | (df["age"] > 100)]
print(f"Invalid ages: {len(invalid_age):,}")
invalid_age[["customer_id", "age"]].head(20)


## 15. Membership Tenure Validation

In [ ]:
print(f"Zero tenure: {(df['membership_years'] == 0).sum():,}")
print(f"Negative tenure: {(df['membership_years'] < 0).sum():,}")


## 16. Transaction Validation

In [ ]:
print(f"Zero transactions: {(df['total_transactions'] == 0).sum():,}")
print(f"Negative transactions: {(df['total_transactions'] < 0).sum():,}")


## 17. Negative Financial Values

In [ ]:
financial_columns = [
    "unit_price", "avg_purchase_value", "avg_transaction_value",
    "total_sales", "total_discounts_received", "total_returned_value"
]

for col in financial_columns:
    if col in df.columns:
        print(f"{col}: {(df[col] < 0).sum():,} negative values")


## 18. Zero-Value Analysis

In [ ]:
important_columns = [
    "total_sales", "total_transactions", "total_items_purchased",
    "website_visits", "online_purchases", "in_store_purchases",
    "customer_support_calls"
]

for col in important_columns:
    if col in df.columns:
        print(f"{col}: {(df[col] == 0).sum():,} zero values")


# Categorical Analysis

## 19. Categorical Value Distributions

In [ ]:
for col in categorical_cols:
    print(f"\n{'=' * 60}")
    print(f"Column: {col}")
    print(f"Unique values: {df[col].nunique():,}")
    print(df[col].value_counts(dropna=False).head(15))


## 20. Check Categorical Whitespace

In [ ]:
for col in categorical_cols:
    if df[col].dtype == "object":
        count = (df[col].astype(str) != df[col].astype(str).str.strip()).sum()
        print(f"{col}: {count:,} values with surrounding whitespace")


# Distribution Analysis

## 21. Histograms for Numerical Features

In [ ]:
df[numeric_cols].hist(figsize=(18, 16), bins=30)
plt.tight_layout()
plt.show()


## 22. Important Customer Features

In [ ]:
important_features = [
    "age", "membership_years", "total_transactions", "total_sales",
    "avg_transaction_value", "purchase_frequency",
    "days_since_last_purchase", "website_visits", "app_usage",
    "customer_support_calls"
]

for col in important_features:
    if col in df.columns:
        plt.figure(figsize=(8, 4))
        plt.hist(df[col].dropna(), bins=30)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Number of Customers")
        plt.tight_layout()
        plt.show()


# Outlier Analysis

## 23. Boxplots for Important Features

Outliers are investigated here but are not automatically removed.

In [ ]:
outlier_features = [
    "total_sales", "total_transactions", "total_items_purchased",
    "avg_transaction_value", "website_visits",
    "customer_support_calls", "days_since_last_purchase"
]

for col in outlier_features:
    if col in df.columns:
        plt.figure(figsize=(8, 4))
        plt.boxplot(df[col].dropna())
        plt.title(f"Boxplot - {col}")
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()


# Behavioral Analysis

## 24. Behavioral Feature Statistics

In [ ]:
behavior_features = [
    "website_visits", "app_usage", "social_media_engagement",
    "online_purchases", "in_store_purchases",
    "customer_support_calls", "days_since_last_purchase"
]
behavior_features = [c for c in behavior_features if c in df.columns]
df[behavior_features].describe().T


# Correlation Analysis

## 25. Numerical Feature Correlation Matrix

In [ ]:
correlation = df[numeric_cols].corr()

plt.figure(figsize=(18, 14))
plt.imshow(correlation, aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=90)
plt.yticks(range(len(correlation.columns)), correlation.columns)
plt.title("Numerical Feature Correlation Matrix")
plt.tight_layout()
plt.show()


# Business Relationship Analysis

## 26. Total Sales vs Total Transactions

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["total_transactions"], df["total_sales"], alpha=0.2)
plt.xlabel("Total Transactions")
plt.ylabel("Total Sales")
plt.title("Total Sales vs Total Transactions")
plt.tight_layout()
plt.show()


## 27. Website Visits vs Total Transactions

In [ ]:
if "website_visits" in df.columns:
    plt.figure(figsize=(8, 5))
    plt.scatter(df["website_visits"], df["total_transactions"], alpha=0.2)
    plt.xlabel("Website Visits")
    plt.ylabel("Total Transactions")
    plt.title("Website Visits vs Total Transactions")
    plt.tight_layout()
    plt.show()


## 28. Online vs In-Store Purchases

In [ ]:
if "online_purchases" in df.columns and "in_store_purchases" in df.columns:
    plt.figure(figsize=(8, 5))
    plt.scatter(df["online_purchases"], df["in_store_purchases"], alpha=0.2)
    plt.xlabel("Online Purchases")
    plt.ylabel("In-Store Purchases")
    plt.title("Online vs In-Store Purchases")
    plt.tight_layout()
    plt.show()


# EDA Quality Report

In [ ]:
print("=" * 70)
print("CUSTOMER SEGMENTATION PLATFORM — EDA QUALITY REPORT")
print("=" * 70)
print(f"Rows:                    {len(df):,}")
print(f"Columns:                 {df.shape[1]:,}")
print(f"Unique customers:        {df['customer_id'].nunique():,}")
print(f"Duplicate rows:          {df.duplicated().sum():,}")
print(f"Duplicate customers:     {df['customer_id'].duplicated().sum():,}")
print(f"Missing cells:           {df.isnull().sum().sum():,}")
print(f"Invalid ages:            {((df['age'] < 18) | (df['age'] > 100)).sum():,}")
print(f"Zero transactions:       {(df['total_transactions'] == 0).sum():,}")
print(f"Negative transactions:   {(df['total_transactions'] < 0).sum():,}")
print(f"Zero tenure:             {(df['membership_years'] == 0).sum():,}")
print(f"Negative tenure:         {(df['membership_years'] < 0).sum():,}")
print("=" * 70)


# EDA Conclusion

The dataset is customer-level and will be evaluated for missing values, duplicates, invalid values, categorical inconsistencies, distributions, outliers, behavioral patterns, and correlations.

Before clustering, numerical features must be scaled. ID-like variables and fields that do not represent meaningful customer behavior should be excluded during feature engineering. Highly correlated or redundant features should also be reviewed.

Outliers should be investigated and handled carefully rather than automatically removed.

**Next stage:** Feature Engineering and Preprocessing.
